In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

# --- Simulation & Grid ---
dt = 0.002         # Timestep
nr = 400           # Number of radial modes (0 to nr-1)
nt = 136           # Number of non-negative azimuthal modes (0 to nt-1)
nx = 256           # Number of non-negative axial modes
ell = 4.0          # Mapping parameter
ell2 = ell**2
zlen = 5.0/3.0 * (2.0 * np.pi) # Axial length

# --- Hyperviscosity Settings ---
p = 8              # Power for hyperviscosity (k^(2p))
p_half = p / 2.0   # Exponent for k^2 input

# --- Original Implementation Parameters ---
nu_r_orig = 1e-8
nu_t_orig = 1e-8
nu_x_orig = 1e-8

# --- New (Normalized) Implementation Parameters ---
D_r_new = 1.0/dt
D_t_new = 1.0/dt
D_x_new = 1.0/dt

# --- Mask Parameters ---
ratio = 2.0 / 3.0
steepness = 3.0

def hyp3mask(k_vals, ratio_in=2.0/3.0, steepness_in=3.0):
    k_mask = np.ones_like(k_vals, dtype=float)
    k_vals_abs = np.abs(k_vals)
    k_max = np.max(k_vals_abs)
    if k_max < 1.0e-12: k_max = 1.0
    tanh_a = np.tanh(steepness_in)
    if abs(tanh_a) < 1.0e-12: tanh_a = 1.0e-12
    ramp_width = 1.0 - ratio_in
    if ramp_width < 1.0e-12:
        k_mask[k_vals_abs / k_max < ratio_in] = 0.0
        k_mask[k_vals_abs / k_max >= ratio_in] = 1.0
        return k_mask
    for i in range(len(k_vals_abs)):
        k_norm = k_vals_abs[i] / k_max
        if k_norm < ratio_in:
            k_mask[i] = 0.0
        else:
            x_ramp = (k_norm - ratio_in) / ramp_width
            y_ramp = steepness_in * (2.0 * x_ramp - 1.0)
            k_mask[i] = (np.tanh(y_ramp) + tanh_a) / (2.0 * tanh_a)
            k_mask[i] = max(0.0, min(1.0, k_mask[i]))
    return k_mask

n_modes = np.arange(nr)
m_modes = np.arange(nt)
k_axial = np.arange(nx) * (2.0 * np.pi / zlen)
kx_max = k_axial[-1] if nx > 0 else 0.0
k_r_sq_op = np.zeros(nr)
for n in range(1, nr):
    k_r_sq_op[n] = (1.5**2 * n * (n + 1)) / ell2
k_t_sq_op = m_modes**2
k_x_sq_op = k_axial**2

mask_r = hyp3mask(np.real(n_modes), ratio_in=ratio, steepness_in=steepness)
mask_t = hyp3mask(np.sqrt(np.maximum(0.0, k_t_sq_op)), ratio_in=ratio, steepness_in=steepness)
mask_x = hyp3mask(np.sqrt(np.maximum(0.0, k_x_sq_op)), ratio_in=ratio, steepness_in=steepness)

# Original
filter_r_orig_unmask = np.exp(-dt * nu_r_orig * (np.maximum(0.0, k_r_sq_op)**p_half))
filter_t_orig_unmask = np.exp(-dt * nu_t_orig * (np.maximum(0.0, k_t_sq_op)**p_half))
filter_x_orig_unmask = np.exp(-dt * nu_x_orig * (np.maximum(0.0, k_x_sq_op)**p_half))

filter_r_orig = np.exp(-dt * mask_r * nu_r_orig * (np.maximum(0.0, k_r_sq_op)**p_half))
filter_t_orig = np.exp(-dt * mask_t * nu_t_orig * (np.maximum(0.0, k_t_sq_op)**p_half))
filter_x_orig = np.exp(-dt * mask_x * nu_x_orig * (np.maximum(0.0, k_x_sq_op)**p_half))

# Normalized
k_max_r_op_sq = max(np.max(k_r_sq_op), 1.0e-12)
k_max_t_op_sq = max(np.max(k_t_sq_op), 1.0e-12)
k_max_x_op_sq = max(np.max(k_x_sq_op), 1.0e-12)
op_r_norm_sq = k_r_sq_op / k_max_r_op_sq
op_t_norm_sq = k_t_sq_op / k_max_t_op_sq
op_x_norm_sq = k_x_sq_op / k_max_x_op_sq

filter_r_new_unmask = np.exp(-dt * D_r_new * (np.maximum(0.0, op_r_norm_sq)**p_half))
filter_t_new_unmask = np.exp(-dt * D_t_new * (np.maximum(0.0, op_t_norm_sq)**p_half))
filter_x_new_unmask = np.exp(-dt * D_x_new * (np.maximum(0.0, op_x_norm_sq)**p_half))

filter_r_new = np.exp(-dt * mask_r * D_r_new * (np.maximum(0.0, op_r_norm_sq)**p_half))
filter_t_new = np.exp(-dt * mask_t * D_t_new * (np.maximum(0.0, op_t_norm_sq)**p_half))
filter_x_new = np.exp(-dt * mask_x * D_x_new * (np.maximum(0.0, op_x_norm_sq)**p_half))

# Ensure filter is exactly 1 for k=0 mode
for filt in [filter_r_orig, filter_t_orig, filter_x_orig, filter_r_new, filter_t_new, filter_x_new]:
    if len(filt) > 0: filt[0] = 1.0

plt.style.use('seaborn-v0_8-darkgrid')
fig, axes = plt.subplots(2, 3, figsize=(16, 9)) # sharey is False
fig.suptitle(f'Hyperviscosity Filter Comparison (p={p}, dt={dt})', fontsize=16)

# --- Helper function for setting y-limits and annotation in legend ---
def setup_plot_legend_clamp(ax, x_modes, filter_vals_masked, filter_vals_unmasked, label_base, color, style):
    # Plot unmasked filter first (dashed line, no markers)
    max_damp_unmask = filter_vals_unmasked[-1]
    unmask_label = f'{label_base} (Unmasked)\nMax Damp: {max_damp_unmask:.2e}'
    ax.semilogy(x_modes, filter_vals_unmasked, color + '--', markersize=0, linewidth=1.5, alpha=0.6, label=unmask_label)
    
    # Plot masked filter (solid line with markers)
    max_damp_val = filter_vals_masked[-1]
    mask_label = f'{label_base} (Masked)\nMax Damp: {max_damp_val:.2e}'
    ax.semilogy(x_modes, filter_vals_masked, color + style, markersize=3, linewidth=1, label=mask_label)

    lower_bound = min(max_damp_val, max_damp_unmask)
    upper_bound = 1.0
    if abs(upper_bound - lower_bound) < 1e-12 * upper_bound:
        lower_bound *= (1.0 - 1e-3)
        upper_bound *= (1.0 + 1e-3)
    if upper_bound < lower_bound: upper_bound = lower_bound * (1 + 1e-3)

    ax.set_ylim(bottom=lower_bound, top=upper_bound)
    ax.grid(True, which='both', linestyle=':')
    ax.legend(loc='best', fontsize='small')

plot_info = [
    {'title': 'Radial', 'xlabel': 'Radial Mode (n)', 'x_data': n_modes, 'color': 'r'},
    {'title': 'Azimuthal', 'xlabel': 'Azimuthal Mode (m)', 'x_data': m_modes, 'color': 'g'},
    {'title': 'Axial', 'xlabel': 'Axial Wavenumber (k_z)', 'x_data': k_axial, 'color': 'b'}
]

# --- Row 1: Original Implementation ---
for i, info in enumerate(plot_info):
    ax = axes[0, i]
    filters_masked = [filter_r_orig, filter_t_orig, filter_x_orig][i]
    filters_unmasked = [filter_r_orig_unmask, filter_t_orig_unmask, filter_x_orig_unmask][i]
    nus = [nu_r_orig, nu_t_orig, nu_x_orig][i]
    setup_plot_legend_clamp(ax, info['x_data'], filters_masked, filters_unmasked, f'Original (nu={nus:.1e})', info['color'], '.-')
    ax.set_title(f"{info['title']} Filter (Original)")
    ax.set_xlabel(info['xlabel'])
    if i == 0: ax.set_ylabel('Filter Value (Log Scale)')

# --- Row 2: New (Normalized) Implementation ---
for i, info in enumerate(plot_info):
    ax = axes[1, i]
    filters_masked = [filter_r_new, filter_t_new, filter_x_new][i]
    filters_unmasked = [filter_r_new_unmask, filter_t_new_unmask, filter_x_new_unmask][i]
    Ds = [D_r_new, D_t_new, D_x_new][i]
    setup_plot_legend_clamp(ax, info['x_data'], filters_masked, filters_unmasked, f'Normalized (D={Ds*dt:.1f})', info['color'], '.-')
    ax.set_title(f"{info['title']} Filter (Normalized)")
    ax.set_xlabel(info['xlabel'])
    if i == 0: ax.set_ylabel('Filter Value (Log Scale)')


plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# --- Print comparison for highest mode ---
highest_k_idx = nx - 1

print(f"ZLEN = {zlen:.4f}")
print(f"Max Axial Wavenumber (k_max) = {kx_max:.4f}")
print("-" * 30)
print(f"Original Filter Values at Highest Modes:")
print(f"  Radial (n={nr-1}): {filter_r_orig[-1]:.3e}")
print(f"  Azimuthal (m={nt-1}): {filter_t_orig[-1]:.3e}")
print(f"  Axial (k_idx={highest_k_idx}, k={k_axial[highest_k_idx]:.2f}): {filter_x_orig[highest_k_idx]:.3e}")
print("-" * 30)
print(f"Normalized Filter Values at Highest Modes:")
print(f"  Radial (n={nr-1}): {filter_r_new[-1]:.3e} (Target: exp(-dt*D_r) ~ {math.exp(-dt*D_r_new):.3e})")
print(f"  Azimuthal (m={nt-1}): {filter_t_new[-1]:.3e} (Target: exp(-dt*D_t) ~ {math.exp(-dt*D_t_new):.3e})")
print(f"  Axial (k_idx={highest_k_idx}, k={k_axial[highest_k_idx]:.2f}): {filter_x_new[highest_k_idx]:.3e} (Target: exp(-dt*D_x) ~ {math.exp(-dt*D_x_new):.3e})")